# Label Construction & Noise Analysis

Defines the target label, finds noise sources, documents assumptions.

Why it matters: bad labels degrade calibration. Mislabeled examples bias PD estimates, which breaks Expected Loss calculations.

In [ ]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv('../data/raw/default_of_credit_card_clients.csv')

# Apply temporal splits from Step 2
train_ids = df[df['ID'] <= 20000]['ID'].values
val_ids = df[(df['ID'] > 20000) & (df['ID'] <= 25000)]['ID'].values
test_ids = df[df['ID'] > 25000]['ID'].values

train_df = df[df['ID'].isin(train_ids)].copy()
val_df = df[df['ID'].isin(val_ids)].copy()
test_df = df[df['ID'].isin(test_ids)].copy()

print(f"Train: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

## Step 1: Label Definition

### Target Variable

Column: `default payment next month`. Binary: 0 = no default, 1 = default.

Definition: Y=1 if default in month 0 (month after observation window), Y=0 otherwise.

### Temporal Alignment

Observation window: months -6 to -1 (what we know at decision time).
Decision point: end of month -1.
Outcome window: month 0 (what we're predicting).
Target: default in month 0, not during observation period.

```
Month -6  -5  -4  -3  -2  -1  |  0
[Observation Window]          | [Outcome]
PAY_6 ... PAY_2 PAY_0         | default payment next month
```

### What "Default" Means

Dataset definition: borrower fails to make minimum payment in month 0. Single-month observation.

Industry typically uses 90+ days past due or charge-off. This definition is looser—just one missed payment qualifies. Creates label noise: borderline cases where payment history suggests default but label says otherwise, or vice versa.

In [ ]:
# Inspect target variable
target_col = 'default payment next month'

print("=== TARGET VARIABLE INSPECTION ===\n")
print(f"Column name: '{target_col}'")
print(f"Data type: {df[target_col].dtype}")
print(f"Unique values: {sorted(df[target_col].unique())}")
print(f"\nValue counts (full dataset):")
print(df[target_col].value_counts().sort_index())
print(f"\nProportions (full dataset):")
print(df[target_col].value_counts(normalize=True).sort_index())
print(f"\nMissing values: {df[target_col].isna().sum()}")

## Step 2: Analyze Label Quality

### Class Balance Across Splits

Check default rates across train/val/test. Why it matters: if default rates drift (22% → 20% → 25%), the population is shifting. That's a problem for model stability.

Large differences might mean the temporal proxy is wrong. Or it might mean real distribution shift. Need to know which.

In [ ]:
# Analyze class balance across splits
target_col = 'default payment next month'

print("=== CLASS BALANCE ACROSS SPLITS ===\n")

for split_name, split_df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    default_rate = split_df[target_col].mean()
    default_count = split_df[target_col].sum()
    total = len(split_df)
    
    print(f"{split_name}:")
    print(f"  Total samples: {total:,}")
    print(f"  Defaults (Y=1): {default_count:,} ({default_rate:.4f} or {default_rate*100:.2f}%)")
    print(f"  Non-defaults (Y=0): {total - default_count:,} ({(1-default_rate):.4f} or {(1-default_rate)*100:.2f}%)")
    print()

# Compare default rates
train_rate = train_df[target_col].mean()
val_rate = val_df[target_col].mean()
test_rate = test_df[target_col].mean()

print("=== DEFAULT RATE COMPARISON ===\n")
print(f"Train default rate:      {train_rate:.4f} ({train_rate*100:.2f}%)")
print(f"Validation default rate: {val_rate:.4f} ({val_rate*100:.2f}%)")
print(f"Test default rate:       {test_rate:.4f} ({test_rate*100:.2f}%)")
print(f"\nDifference (Val - Train): {val_rate - train_rate:.4f} ({(val_rate - train_rate)*100:.2f} percentage points)")
print(f"Difference (Test - Train): {test_rate - train_rate:.4f} ({(test_rate - train_rate)*100:.2f} percentage points)")
print(f"Difference (Test - Val):   {test_rate - val_rate:.4f} ({(test_rate - val_rate)*100:.2f} percentage points)")

### Stability of Default Rate Over Temporal Split

Interpretation:
- Stable rates (within ~1-2pp): consistent distribution, supports ID-as-proxy assumption.
- Increasing rates: deteriorating credit quality, economic downturn, riskier portfolio.
- Decreasing rates: improving quality, tighter underwriting.
- Large fluctuations (>5pp): temporal proxy might be wrong, or real shift.

For PD estimation: stable rates are better—consistent risk distribution. Large shifts need calibration adjustment.

### Edge Cases: Borderline Repayment Behavior

Borderline cases where labels might be wrong:

1. Paid late but no default (Y=0): PAY_0 > 0 (recent delay) but made payment in month 0. Should this be Y=1?
2. Defaulted with good history (Y=1): PAY_0 <= 0 (good payment history) but defaulted in month 0. Sudden default or labeling error?
3. Inconsistent patterns: mixed signals that make prediction ambiguous.

These create label noise. If they're common, calibration degrades.